<a href="https://colab.research.google.com/github/minbj1226/pytorch-basics/blob/main/04_Training_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Training Loop란 무엇인가?
- Training Loop: 데이터 배치를 반복해서 모델에 입력하고, 손실을 계산한 뒤, 역전파와 옵티마이저 업데이트를 통해 파라미터를 수정하는 학습 반복 과정이다.

### 2. Training Loop의 전체 흐름
1. 모델에 데이터 입력
2. 예측값 계산
3. 손실함수 계산
4. ```optimizer.zero_grad()``` 실행
5. ```loss.backward()``` 실행
6. ```optimizer.step()``` 실행

### 3. Training Loop에 필요한 구성 요소
- model: 입력 데이터를 받아 예측값을 출력하는 모델
- dataloader: 학습 데이터를 batch 단위로 불러오는 반복 가능한 객체
- loss function: 예측값과 정답값의 차이를 확인하는 함수
- optimizer: 계산된 gradient를 바탕으로 모델 파라미터를 업데이트 하는 도구
- epoch: 전체 데이터셋을 몇 번 반복하여 학습할지 나타내는 횟수

### 4. `zero_grad()`, `backward()`, `step()`의 역할
- ```zero_grad()```: 이전 step에서 누적된 gradient 값을 초기화하는 역할
- ```backward()```: loss를 기준으로 각 파라미터에 대한 gradient를 계산하는 역할
- ```step()```: 계산된 gradient를 이용해 파라미터를 업데이트 하는 역할

### 5. Epoch와 Batch의 의미
- Epoch: 전체 학습 데이터를 한 번 모두 사용하여 학습하는 단위
- Batch: 전체 데이터를 한 번에 처리하지 않고, 여러 개의 작은 묶음으로 나누어 학습할 때의 한 묶음을 의미

### 6. Train Loop와 Test Loop의 차이
- Train Loop: 손실을 계산한 뒤 역전파와 optimizer step을 통해 파라미터를 업데이트하면서 학습을 진행하는 과정
- Test Loop: 파라미터를 업데이트하지 않고, 학습된 모델의 성능을 평가하기 위해 손실과 정확도를 계산하는 과정

In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [ ]:
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

In [ ]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([6])


In [ ]:
input_images = torch.rand(3, 28, 28)
print(input_images.size())

torch.Size([3, 28, 28])


In [ ]:
flatten = nn.Flatten()
flat_image = flatten(input_images)
print(flat_image.size())

torch.Size([3, 784])


In [ ]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.0302,  0.2650,  0.1228,  0.5255, -0.1509,  0.1726,  0.0418,  0.0410,
          0.2075,  0.0248,  0.4886,  0.2805, -0.1474, -0.3731, -0.1585, -0.0074,
          0.3325, -0.2467, -0.4576, -0.2950],
        [ 0.0050,  0.1036,  0.2283,  0.7813, -0.0410,  0.0386, -0.3602, -0.0521,
          0.3813, -0.0311,  0.5751,  0.1604, -0.6340, -0.0733, -0.2145,  0.1969,
          0.5030, -0.3685, -0.1007, -0.0490],
        [ 0.0776,  0.3114,  0.2248,  0.6336, -0.4262,  0.3086, -0.0258, -0.1001,
          0.1005, -0.1166,  0.3855, -0.1420, -0.0116, -0.4103, -0.3170,  0.0038,
          0.2156, -0.3106, -0.3489,  0.0251]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.2650, 0.1228, 0.5255, 0.0000, 0.1726, 0.0418, 0.0410, 0.2075,
         0.0248, 0.4886, 0.2805, 0.0000, 0.0000, 0.0000, 0.0000, 0.3325, 0.0000,
         0.0000, 0.0000],
        [0.0050, 0.1036, 0.2283, 0.7813, 0.0000, 0.0386, 0.0000, 0.0000, 0.3813,
         0.0000, 0.5751, 0.1604, 0.0000, 0.0000, 0.00

In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)

input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

In [ ]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [ ]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0318, -0.0184,  0.0252,  ..., -0.0142, -0.0332,  0.0249],
        [-0.0064,  0.0189,  0.0226,  ...,  0.0258,  0.0155,  0.0341]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0144, 0.0045], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0213,  0.0201,  0.0047,  ...,  0.0092, -0.0138, -0.0333],
        [-0.0295,  0.0411, -0.0124,  ...,  0.0394,  0.0219, -0.0113]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

### Hyperparameters
- Number of Epochs
- Batch Size
- Learning Rate

In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
learning_rate = 0.001
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch, (X, y) in enumerate(dataloader):
    pred = model(X)
    loss = loss_fn(pred, y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), (batch + 1) * len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
  model.eval()
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0

  with torch.no_grad():
    for X, y in dataloader:
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)

print("Done!")

Epoch 1
-------------------------------
loss: 2.292510 [   64/60000]
loss: 2.277329 [ 6464/60000]
loss: 2.262749 [12864/60000]
loss: 2.261201 [19264/60000]
loss: 2.242398 [25664/60000]
loss: 2.222247 [32064/60000]
loss: 2.225530 [38464/60000]
loss: 2.195569 [44864/60000]
loss: 2.184055 [51264/60000]
loss: 2.152548 [57664/60000]
Test Error: 
 Accuracy: 52.8%, Avg loss: 2.149202 

Epoch 2
-------------------------------
loss: 2.155895 [   64/60000]
loss: 2.140981 [ 6464/60000]
loss: 2.086328 [12864/60000]
loss: 2.098091 [19264/60000]
loss: 2.055228 [25664/60000]
loss: 1.997495 [32064/60000]
loss: 2.014987 [38464/60000]
loss: 1.941985 [44864/60000]
loss: 1.937188 [51264/60000]
loss: 1.854095 [57664/60000]
Test Error: 
 Accuracy: 60.0%, Avg loss: 1.861996 

Epoch 3
-------------------------------
loss: 1.896278 [   64/60000]
loss: 1.859882 [ 6464/60000]
loss: 1.746242 [12864/60000]
loss: 1.778974 [19264/60000]
loss: 1.679881 [25664/60000]
loss: 1.629738 [32064/60000]
loss: 1.639321 [38464/